<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/muse_wonder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json

def list_openrouter_models():
    """
    Connects to the OpenRouter API to fetch the list of available models
    and prints key details for each model.
    """

    # OpenRouter API endpoint for listing models
    API_URL = "https://openrouter.ai/api/v1/models"

    print(f"Fetching models from: {API_URL}\n")

    try:
        # Send a GET request to the models endpoint
        response = requests.get(API_URL)

        # Raise an exception for bad status codes (4xx or 5xx)
        response.raise_for_status()

        # Parse the JSON response
        data = response.json()

        # The list of models is contained in the 'data' field
        models = data.get('data', [])

        if not models:
            print("No models found in the API response.")
            return

        # --- Print the results in a formatted way ---
        print(f"Found {len(models)} total models. Key details:\n")
        print("{:<45} {:<30} {:<15}".format("Model ID", "Name", "Context Length"))
        print("-" * 90)

        # Iterate over the model list and print details
        for model in models:
            model_id = model.get('id', 'N/A')
            name = model.get('name', 'N/A')
            context_length = model.get('context_length', 'N/A')

            # Truncate model ID and name for clean printing
            display_id = model_id[:42] + '...' if len(model_id) > 45 else model_id
            display_name = name[:27] + '...' if len(name) > 30 else name

            print("{:<45} {:<30} {:<15}".format(display_id, display_name, context_length))

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while connecting to the OpenRouter API: {e}")
    except json.JSONDecodeError:
        print("Error: Failed to decode JSON response from the API.")

if __name__ == "__main__":
    list_openrouter_models()

In [ ]:
!pip install pyautogen -q
!pip install autogen -q

In [ ]:
!pip install --upgrade "protobuf>=6.33.5" "google-api-core>=2.24.0"

## case1

In [3]:
from openai import OpenAI
import os

# Secure API key retrieval using Colab secrets
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if not OPENROUTER_API_KEY:
        raise ValueError("OPENROUTER_API_KEY not found in Colab secrets.")
except ImportError:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY")
except Exception as e:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY")

# Initialize the OpenRouter client with Muse Wonder
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Send your direct prompt to Meta's Muse Wonder
response = client.chat.completions.create(
    model="meta/muse-wonder",
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant powered by Meta's Muse Wonder."},
        {"role": "user", "content": "Explain how your 1-million-token context window handles multi-file codebases."}
    ],
    temperature=0.7,
)

print(response.choices[0].message.content)

With a 1-million-token context window, I don't treat a codebase like a single long document — I treat it like a loaded project in an IDE. Here's how that works in practice:

### 1. What 1M tokens actually holds for code

1 token is roughly 3-4 characters of code. So 1M tokens is about:

* ~60,000 - 80,000 lines of typical code
* ~750,000 words of docs + code combined
* A medium-sized repo (10-30 MB of source) in full, or the *relevant slice* of a much larger monorepo

That's enough to stop aggressively chunking everything. With a 128k window, I often only see the file you mentioned + a few retrieved snippets. With 1M, I can keep the file you're editing, all its importers/importees, its tests, its types, and its config in attention at the same time.

### 2. How multi-file projects are loaded

Files aren't just dumped randomly. They're concatenated with structure:

```
[FILE: src/auth/login.ts]
...
[FILE: src/auth/session.ts]
...
[FILE: src/db/user.ts]
...
```

That header matters. It le

## AGENTIC

In [4]:
import os
import json
import subprocess
from openai import OpenAI

# 1. Secure API key retrieval (works in Google Colab or local environment)
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if not OPENROUTER_API_KEY:
        raise ValueError("OPENROUTER_API_KEY not found in Colab secrets.")
except ImportError:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY")

# 2. Initialize OpenRouter client pointing to Meta's Muse Wonder
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# 3. Define the Tools available to the Agent
tools = [
    {
        "type": "function",
        "function": {
            "name": "list_workspace",
            "description": "List all files and folders in the workspace directory.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a specific file in the workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string", "description": "The path of the file to read."}
                },
                "required": ["file_path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "execute_python_code",
            "description": "Execute a python script and return its stdout and stderr.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string", "description": "The Python code to execute."}
                },
                "required": ["code"]
            }
        }
    }
]

# Implement the actual tool logic
def list_workspace():
    os.makedirs("agent_workspace", exist_ok=True)
    return json.dumps(os.listdir("agent_workspace"))

def read_file(file_path):
    full_path = os.path.join("agent_workspace", file_path)
    if os.path.exists(full_path):
        with open(full_path, "r") as f:
            return f.read()
    return f"Error: File {file_path} not found."

def execute_python_code(code):
    os.makedirs("agent_workspace", exist_ok=True)
    script_path = os.path.join("agent_workspace", "temp_agent_script.py")
    with open(script_path, "w") as f:
        f.write(code)
    try:
        result = subprocess.run(
            ["python", script_path],
            capture_output=True,
            text=True,
            timeout=30
        )
        output = f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        return output
    except Exception as e:
        return f"Execution failed: {str(e)}"

# Map tool names to python functions
available_tools = {
    "list_workspace": list_workspace,
    "read_file": read_file,
    "execute_python_code": execute_python_code
}

# 4. The Agentic Loop
def run_agentic_task(prompt):
    messages = [
        {"role": "system", "content": "You are an autonomous engineering agent powered by Meta's Muse Wonder. You have access to local tools to inspect files, write code, and execute scripts. Solve the user's task step-by-step using your tools."},
        {"role": "user", "content": prompt}
    ]

    print(f"--- Starting Agentic Task ---")

    for step in range(6):  # Max 6 reasoning/tool turns
        response = client.chat.completions.create(
            model="meta/muse-wonder",
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.2
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        # Check if the model wants to call a tool
        if response_message.tool_calls:
            print(f"\n[Turn {step+1}] Agent decided to call tools:")
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                print(f"  -> Tool: {function_name} | Args: {function_args}")

                # Execute the tool
                tool_to_call = available_tools.get(function_name)
                if tool_to_call:
                    tool_output = tool_to_call(**function_args)
                else:
                    tool_output = f"Error: Tool {function_name} not found."

                # Feed the tool result back into the conversation
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": tool_output
                })
        else:
            # If no tool call, the agent has finished its thought process and returned text
            print(f"\n[Task Complete]\n{response_message.content}")
            break

# 5. Kick off a real task for the agent
task_prompt = """
Create a Python script inside our workspace named 'data_generator.py' that generates a NumPy array of 1,000 random floating-point numbers, computes their statistical mean and standard deviation, and saves the summary into a text file called 'stats_report.txt'.
Then, execute the script using your tools and read back the contents of 'stats_report.txt' to verify the output.
"""

run_agentic_task(task_prompt)

--- Starting Agentic Task ---

[Turn 1] Agent decided to call tools:
  -> Tool: list_workspace | Args: {}

[Turn 2] Agent decided to call tools:
  -> Tool: execute_python_code | Args: {'code': 'script_content = \'\'\'import numpy as np\n\ndef main():\n    # Generate a NumPy array of 1,000 random floating-point numbers\n    # Set seed for reproducibility (optional, but ensures consistent results)\n    np.random.seed(42)\n    data = np.random.randn(1000)  # Standard normal distribution; alternatively np.random.rand(1000)\n\n    # Compute statistical mean and standard deviation\n    mean = np.mean(data)\n    std_dev = np.std(data)\n\n    # Prepare summary\n    summary = (\n        "Statistical Summary Report\\\\n"\n        "==========================\\\\n"\n        f"Number of samples: {len(data)}\\\\n"\n        f"Mean: {mean:.6f}\\\\n"\n        f"Standard Deviation: {std_dev:.6f}\\\\n"\n    )\n\n    # Save summary into a text file\n    with open("stats_report.txt", "w") as f:\n        f.

In [5]:
import os
import json
import subprocess
from openai import OpenAI

# 1. Secure API key retrieval
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if not OPENROUTER_API_KEY:
        raise ValueError("OPENROUTER_API_KEY not found in Colab secrets.")
except ImportError:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Define tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "list_workspace",
            "description": "List all files and folders in the agent_workspace directory.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a specific file from agent_workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string", "description": "The name of the file to read."}
                },
                "required": ["file_path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "execute_python_code",
            "description": "Execute a python script inside the agent_workspace directory and return its output.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string", "description": "The Python code to execute."}
                },
                "required": ["code"]
            }
        }
    }
]

# Enforce strict workspace isolation
WORKSPACE_DIR = "agent_workspace"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

def list_workspace():
    return json.dumps(os.listdir(WORKSPACE_DIR))

def read_file(file_path):
    # Strip any directory prefix if the agent accidentally included it
    clean_name = os.path.basename(file_path)
    full_path = os.path.join(WORKSPACE_DIR, clean_name)
    if os.path.exists(full_path):
        with open(full_path, "r") as f:
            return f.read()
    return f"Error: File {clean_name} not found in {WORKSPACE_DIR}."

def execute_python_code(code):
    script_path = os.path.join(WORKSPACE_DIR, "temp_agent_script.py")
    with open(script_path, "w") as f:
        f.write(code)
    try:
        # Run subprocess explicitly setting cwd to WORKSPACE_DIR so files are created there locally
        result = subprocess.run(
            ["python", "temp_agent_script.py"],
            cwd=WORKSPACE_DIR,
            capture_output=True,
            text=True,
            timeout=30
        )
        output = f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        return output
    except Exception as e:
        return f"Execution failed: {str(e)}"

available_tools = {
    "list_workspace": list_workspace,
    "read_file": read_file,
    "execute_python_code": execute_python_code
}

# 2. Agentic Loop with strict path instructions
def run_agentic_task(prompt):
    messages = [
        {"role": "system", "content": f"You are an autonomous engineering agent powered by Meta's Muse Wonder. All your file operations and python scripts execute inside the '{WORKSPACE_DIR}' directory. Always write files directly to the current working directory without nested path prefixes."},
        {"role": "user", "content": prompt}
    ]

    print(f"--- Starting Agentic Task ---")

    for step in range(6):
        response = client.chat.completions.create(
            model="meta/muse-wonder",
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.2
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            print(f"\n[Turn {step+1}] Agent decided to call tools:")
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                print(f"  -> Tool: {function_name} | Args: {function_args}")

                tool_to_call = available_tools.get(function_name)
                if tool_to_call:
                    tool_output = tool_to_call(**function_args)
                else:
                    tool_output = f"Error: Tool {function_name} not found."

                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": tool_output
                })
        else:
            print(f"\n[Task Complete]\n{response_message.content}")
            break

task_prompt = """
Create a Python script named 'data_generator.py' that generates a NumPy array of 1,000 random floating-point numbers, computes their statistical mean and standard deviation, and saves the summary into a text file called 'stats_report.txt'.
Then, execute the script using your tools and read back the contents of 'stats_report.txt' to verify the output.
"""

run_agentic_task(task_prompt)

--- Starting Agentic Task ---

[Turn 1] Agent decided to call tools:
  -> Tool: list_workspace | Args: {}

[Turn 2] Agent decided to call tools:
  -> Tool: execute_python_code | Args: {'code': 'script_content = \'\'\'"""\ndata_generator.py\nGenerates a NumPy array of 1,000 random floating-point numbers,\ncomputes their statistical mean and standard deviation,\nand saves the summary into a text file called \'stats_report.txt\'.\n"""\n\nimport numpy as np\n\n\ndef main():\n    # Set a seed for reproducibility (optional, ensures consistent results)\n    np.random.seed(42)\n\n    # Generate a NumPy array of 1,000 random floating-point numbers\n    data = np.random.random(1000)  # Uniform distribution in [0.0, 1.0)\n\n    # Compute statistical mean and standard deviation\n    mean = np.mean(data)\n    std_dev = np.std(data)\n\n    # Prepare summary\n    summary = (\n        "Statistical Summary Report\\\\n"\n        "==========================\\\\n"\n        f"Count: {len(data)}\\\\n"\n    

## MUSE APP SIMULATION

In [6]:
import os
import json
import subprocess
from openai import OpenAI

# 1. Secure API key retrieval (Colab or local)
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if not OPENROUTER_API_KEY:
        raise ValueError("OPENROUTER_API_KEY not found in Colab secrets.")
except ImportError:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY")

# Initialize client for Muse Wonder via OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

WORKSPACE = "muse_app_workspace"
os.makedirs(WORKSPACE, exist_ok=True)

def call_muse_wonder(system_prompt, user_prompt):
    """Helper function to invoke Muse Wonder with specific system instructions."""
    response = client.chat.completions.create(
        model="meta/muse-wonder",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

# 2. Simulate Muse App Multi-Agent Coordination Pipeline
def run_muse_app_simulation(project_goal):
    print(f"=== [MUSE APP SIMULATION INITIALIZED] ===")
    print(f"Goal: {project_goal}\n")

    # Phase 1: Planner Agent orchestrates the workstream
    print("--- Phase 1: Planner Agent (Structuring Workstream) ---")
    planner_sys = "You are the lead Muse Planner agent. Analyze the project goal and break it down into 2 distinct parallel worker tasks (Module A and Module B) and 1 review task."
    plan = call_muse_wonder(planner_sys, project_goal)
    print(plan)
    print("\n" + "="*50 + "\n")

    # Phase 2: Parallel Workers (Simulating fanned-out subagents)
    print("--- Phase 2: Parallel Workers (Executing Subtasks) ---")

    worker_a_prompt = f"Based on this plan:\n{plan}\nWrite the complete code for Module A (e.g., a data ingestion/processing module) and save it as 'module_a.py'."
    worker_a_res = call_muse_wonder("You are a parallel engineering worker in the Muse ecosystem. Output clean executable Python code.", worker_a_prompt)

    # Save Module A output locally
    with open(os.path.join(WORKSPACE, "module_a.py"), "w") as f:
        f.write(worker_a_res.replace("```python", "").replace("```", ""))
    print("-> Worker A finished: generated module_a.py")

    worker_b_prompt = f"Based on this plan:\n{plan}\nWrite the complete code for Module B (e.g., an analytics/reporting module that imports module_a) and save it as 'module_b.py'."
    worker_b_res = call_muse_wonder("You are a parallel engineering worker in the Muse ecosystem. Output clean executable Python code.", worker_b_prompt)

    # Save Module B output locally
    with open(os.path.join(WORKSPACE, "module_b.py"), "w") as f:
        f.write(worker_b_res.replace("```python", "").replace("```", ""))
    print("-> Worker B finished: generated module_b.py")
    print("\n" + "="*50 + "\n")

    # Phase 3: Background Reviewer Agent audits code quality and security
    print("--- Phase 3: Background Reviewer Agent (Auditing & Testing) ---")

    # Read generated files back for review
    with open(os.path.join(WORKSPACE, "module_a.py"), "r") as f:
        code_a = f.read()
    with open(os.path.join(WORKSPACE, "module_b.py"), "r") as f:
        code_b = f.read()

    review_prompt = f"""
    Review the following two modules generated for the project:

    --- MODULE A ---
    {code_a}

    --- MODULE B ---
    {code_b}

    Check for syntax correctness, import compatibility, and edge cases. Provide a formal audit verdict and any necessary patch recommendations.
    """

    review_res = call_muse_wonder("You are the senior background reviewer and safety auditor in the Muse ecosystem.", review_prompt)
    print(review_res)
    print("\n=== [MUSE APP SIMULATION COMPLETE] ===")

# 3. Kick off the simulation
target_goal = "Build a multi-file Python pipeline where Module A generates a dataset of simulated user telemetry, and Module B aggregates and logs metrics to disk."
run_muse_app_simulation(target_goal)

=== [MUSE APP SIMULATION INITIALIZED] ===
Goal: Build a multi-file Python pipeline where Module A generates a dataset of simulated user telemetry, and Module B aggregates and logs metrics to disk.

--- Phase 1: Planner Agent (Structuring Workstream) ---
# Project Plan: Multi-File Python Telemetry Pipeline

**Goal:** Build a reproducible pipeline where simulated user telemetry is generated, then aggregated into metrics and logged to disk.

To allow parallel work, the **shared data contract is frozen first**. Module A and Module B can then be built independently against that contract.

## 0. Shared Contract - Lead / Freeze Before Parallel Work

No worker should change this without notifying the other.

**File Layout:**
```
project/
├── src/
│   ├── __init__.py
│   ├── telemetry_generator.py    # Module A
│   └── metrics_aggregator.py     # Module B
├── data/
│   └── telemetry.jsonl           # Generated output
├── output/
│   └── metrics.json              # Aggregated output
├── logs/
│ 